# External Validation: Adult Income Classifier on ACS California (2018)

This notebook takes the champion random forest trained on the 1994 Adult census
extract and evaluates it, unchanged, on a contemporary external cohort: the 2018
American Community Survey 1-year person file for California, pulled through
`folktables` with the `ACSIncome` task definition.

The design follows a standard external validation layout:

1. Load the frozen internal test set and the champion model from the registry.
2. Pull the external cohort and harmonize its schema to the model's feature space.
3. Score both cohorts with the same model and the same operating threshold.
4. Compare discrimination, precision-recall, calibration, and threshold behavior
   side by side.

Nothing is refit here. Every number reflects the model exactly as it was
selected on internal validation.

## Setup

In [ ]:
import os

import numpy as np
import pandas as pd

from folktables import ACSDataSource, ACSIncome

from model_metrics import (
    align_features,
    summarize_model_performance,
)
from model_metrics.model_registry import best_per_algo, load_best_per_algo

In [ ]:
DATA_PATH = "../model_files/"
IMAGE_PATH = "../model_files/images/svg_images/"

SURVEY_YEAR = "2018"
STATE = "CA"

INT_TITLE = "Adult 1994 (internal)"
EXT_TITLE = "ACS CA 2018 (external)"

SELECTION_METRIC = "valid AUC ROC"

## 1. Internal test set

The held-out Adult test split, written to parquet at training time. This is the
reference point every external number is read against.

In [ ]:
X_test = pd.read_parquet(os.path.join(DATA_PATH, "X_test.parquet"))
y_test = pd.read_parquet(os.path.join(DATA_PATH, "y_test.parquet"))

y_adult_test = np.asarray(y_test).astype(int).ravel()

print(f"internal test shape: {X_test.shape}")
print(f"positive rate: {y_adult_test.mean():.3f}")

## 2. External cohort

`ACSIncome` defines the label as personal income above $50,000, matching the
Adult target definition. The first call downloads and caches the raw survey
file, so subsequent runs are local.

In [ ]:
ds = ACSDataSource(survey_year=SURVEY_YEAR, horizon="1-Year", survey="person")
ca = ds.get_data(states=[STATE], download=True)

X_acs, y_acs, group = ACSIncome.df_to_pandas(ca)

y_acs_int = np.asarray(y_acs["PINCP"]).astype(int).ravel()

print(f"external cohort shape: {X_acs.shape}")
print(f"positive rate: {y_acs_int.mean():.3f}")

## 3. Champion model

`best_per_algo` ranks every registered run by the selection metric. The random
forest is carried forward as the primary model; the decision tree and logistic
regression champions stay available in `champs` if a sensitivity comparison is
wanted later.

In [ ]:
best_per_algo(metric=SELECTION_METRIC)

In [ ]:
champs = load_best_per_algo(metric=SELECTION_METRIC)
model_rf = champs["rf_income"]

# operating threshold carried over from internal tuning
threshold = next(iter(model_rf.threshold.values()))
print(f"operating threshold: {threshold}")

In [ ]:
from model_metrics import get_expected_features

get_expected_features(model_rf)

## 4. Feature harmonization

The ACS and Adult schemas describe the same constructs under different names.
`align_features` renames the overlapping columns, drops what the model never
saw, and fills any missing expected column so the frame matches the training
feature order exactly.

| ACS | Adult | Construct |
| --- | --- | --- |
| `AGEP` | `age` | age in years |
| `SCHL` | `education-num` | educational attainment |
| `WKHP` | `hours-per-week` | usual hours worked per week |

Two caveats attach to this mapping and both belong in any write-up of the
results:

- `SCHL` runs on a 1 to 24 attainment scale while `education-num` runs 1 to 16.
  The ordering is preserved and the direction of effect is the same, though the
  spacing differs, so the education contribution is approximate.
- The $50,000 cut point is nominal in both datasets. Twenty-four years of
  inflation means the 2018 threshold is far easier to clear, which inflates the
  external positive rate and will show up directly in the calibration panel.

In [ ]:
ext = align_features(
    X_acs,
    model=model_rf,
    col_map={
        "AGEP": "age",
        "SCHL": "education-num",
        "WKHP": "hours-per-week",
    },
    copy_passthrough=True,
    on_unmapped="warn",
)

ext.head()

## 5. Scoring

One model, one threshold, two cohorts.

In [ ]:
p_adult_test = model_rf.predict_proba(X_test)[:, 1]
p_acs = model_rf.predict_proba(ext)[:, 1]

## 6. Overall performance

Both tables are computed at the internal operating threshold, so the
classification metrics are directly comparable.

In [ ]:
summarize_model_performance(
    y_prob=p_adult_test,
    y=y_adult_test,
    model_title=INT_TITLE,
    return_df=True,
    decimal_places=3,
    model_threshold=threshold,
)

In [ ]:
summarize_model_performance(
    y_prob=p_acs,
    y=y_acs_int,
    model_title=EXT_TITLE,
    return_df=True,
    decimal_places=3,
    model_threshold=threshold,
)

## 7. Subgroup performance

`RAC1P` is the ACS recoded race variable. Stratifying at the same threshold
shows whether the transported model degrades unevenly across groups, which is
the question that matters more than the pooled external AUC.

In [ ]:
summarize_model_performance(
    y_prob=p_acs,
    y=y_acs_int,
    model_title=EXT_TITLE,
    return_df=True,
    decimal_places=3,
    model_threshold=threshold,
    group_category=group["RAC1P"],
)

## 8. Combined validation figure

Four panels on shared axes: ROC, precision-recall, calibration, and threshold
sweep. Internal in black, external in burnt orange.

In [ ]:
probs = [p_adult_test, p_acs]
truths = [y_adult_test, y_acs_int]
titles = [INT_TITLE, EXT_TITLE]

styles = {
    INT_TITLE: {"color": "black", "linewidth": 1.5},
    EXT_TITLE: {"color": "#C1440E", "linewidth": 1.5},
}

SHARED = {
    "y_prob": probs,
    "y": truths,
    "model_title": titles,
    "overlay": True,
    "curve_kwgs": styles,
}

## Limitations

- **Temporal drift.** A 1994 training distribution scored against 2018 data,
  with a nominal dollar threshold held fixed across both. Expect the external
  calibration curve to sit below the diagonal.
- **Geographic restriction.** California only. The Adult extract is national, so
  the comparison confounds time with place.
- **Partial feature overlap.** Three continuous features carry the mapping.
  Occupation, marital status, and the remaining categoricals either differ in
  encoding or are absent, so `align_features` is imputing or dropping them.
  Discrimination in the external cohort should be read as a floor.
- **Survey weights ignored.** ACS person records carry replicate weights
  (`PWGTP`) that are dropped here. Population-level estimates would need them.
- **Race coding.** `RAC1P` categories are not interchangeable with the Adult
  race variable, so subgroup results are internal to the external cohort and
  cannot be compared group-for-group against internal test.